[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gnoejh/AIBookGitHub/blob/main/15_capability_integration.ipynb)

# Layer 5: Capability Integration - Model Context Protocol (MCP)



## Position in the AI System Hierarchy

| Layer | Name | Description |
|-------|------|-------------|
| **Layer 6** | COMMUNICATION | Formatting, messaging protocols |
| **Layer 5** | CAPABILITY INTEGRATION **(THIS)** | MCP, tools, external systems |
| **Layer 4** | COORDINATION | Multi-agent orchestration |
| **Layer 3** | MEMORY | State management |
| **Layer 2** | INVOCATION | API abstraction |
| **Layer 1** | EXECUTION | Model providers |

**Layer 5** bridges AI agents with external capabilities through standardized protocols. MCP (Model Context Protocol) is the primary standard for tool and resource discovery/invocation.

---

## Learning Path Overview

This notebook follows a structured learning path:

1. **Theory & Concepts** (5.1-5.2): Understand what MCP is and why it exists
2. **Hands-on Learning** (5.3-5.5): Build a simple MCP-like system to understand the concepts
3. **Production MCP** (5.6): Learn the real MCP specification and JSON-RPC protocol
4. **Practical Setup** (5.7): Configure MCP servers in IDEs and use programmatically
5. **Complete Architecture** (5.8): Understand the full protocol stack and system integration
6. **Practice** (5.9): Exercises to reinforce learning
7. **Summary & Comparison** (5.10-5.11): Review key concepts and compare with related technologies

---

## 5.1 Theoretical Foundation: MCP Architecture

### 5.1.1 Protocol Stack Hierarchy

MCP operates at the **capability integration layer**, providing a standardized interface between:

1. **Clients** (Agents, IDEs, Runtimes) - Entities that need tools/resources
2. **Servers** (Tool Providers, Resource Providers) - Entities that expose capabilities
3. **Transport Layer** - Communication mechanism (stdio, WebSocket, HTTP)
4. **Protocol Layer** - JSON-RPC 2.0 message format
5. **Semantic Layer** - Tool schemas, resource URIs, capability negotiation

### 5.1.2 Core Theoretical Concepts

| Concept        | Definition                                                      | Role in Stack    |
|----------------|-----------------------------------------------------------------|------------------|
| **Client**     | Agent/IDE/runtime requesting capabilities                      | Consumer layer   |
| **Server**     | Process exposing tools/resources                                | Provider layer   |
| **Capabilities** | Negotiated features (tools, resources, prompts)                | Contract layer   |
| **Tools**      | Executable functions with JSON Schema signatures               | Functional layer |
| **Resources**  | Data sources with URI identifiers                              | Data layer       |
| **Transport**  | Communication channel (stdio/WebSocket/HTTP)                   | Network layer    |
| **Protocol**   | JSON-RPC 2.0 message envelope                                  | Message layer    |

### 5.1.3 Protocol Flow (Theoretical)

```mermaid
sequenceDiagram
    participant Client as Client<br/>(Agent/IDE)
    participant Server as Server<br/>(Provider)
    
    Client->>Server: 1. initialize(clientInfo, capabilities)
    Server-->>Client: 2. result(serverInfo, capabilities)
    
    Client->>Server: 3. tools/list()
    Server-->>Client: 4. result([tool schemas])
    
    Client->>Server: 5. tools/call(name, arguments)
    Server-->>Client: 6. result({ok, result}) or error
```

---

## 5.2 MCP Ecosystem: Systematic Overview

### 5.2.1 Origin & Standardization

**Anthropic** introduced MCP (Model Context Protocol) as an open standard to:
- Standardize tool/resource discovery and invocation
- Enable plug-and-play capability integration
- Support capability negotiation and secure scoping
- Decouple agent implementations from tool providers

### 5.2.2 Client Ecosystem (IDEs & Runtimes)

| Client Type      | Examples                                          | Role                                    |
|------------------|---------------------------------------------------|-----------------------------------------|
| **Web Interfaces** | Claude.ai (Anthropic)                             | Web-based AI assistant with MCP support |
| **IDEs**         | VS Code, Cursor, Claude Code, JetBrains IDEs     | Developer tools integrating MCP servers  |
| **Agent Runtimes** | Custom orchestrators, LangChain, AutoGen          | Runtime environments for AI agents      |
| **CLI Tools**    | MCP CLI, custom clients                           | Command-line interfaces                 |

**Key Client Integrations:**
- **Claude.ai**: Anthropic's web interface with native MCP support for tool integration
- **Claude Code**: Anthropic's IDE integration supporting MCP servers
- **VS Code**: MCP extension for connecting to MCP servers
- **Cursor**: Native MCP support for AI-assisted development
- **JetBrains IDEs**: IntelliJ IDEA, PyCharm, WebStorm, and other JetBrains IDEs with MCP support
- All enable developers and users to connect to MCP servers for enhanced capabilities

### 5.2.3 Server Ecosystem (Providers & Hubs)

| Provider Type    | Examples                                          | Purpose                                    |
|------------------|---------------------------------------------------|--------------------------------------------|
| **Official Hubs** | GitHub (mcp servers), MCP Hubs                    | Centralized repositories of MCP servers    |
| **Marketplaces** | Smithery, mcp.so, mcpservices.com                 | Discovery platforms for MCP servers        |
| **Enterprise**   | Custom internal servers                           | Organization-specific capability providers |

**Provider Categories:**
1. **GitHub**: Hosts official and community MCP servers
2. **Smithery**: MCP server marketplace and discovery
3. **mcp.so**: MCP server registry and documentation
4. **mcpservices.com**: Commercial MCP server marketplace
5. **MCP Hubs**: Community-curated collections

### 5.2.4 Ecosystem Architecture Diagram

```mermaid
graph TB
    subgraph Clients["MCP Clients"]
        CA[Claude.ai<br/>Web Client]
        VC[VS Code<br/>IDE Client]
        CU[Cursor<br/>IDE Client]
        CC[Claude Code<br/>IDE Client]
        JB[JetBrains IDEs<br/>IDE Clients]
    end
    
    subgraph Protocol["MCP Protocol Layer"]
        MCP[initialize<br/>tools/list<br/>tools/call<br/>resources/list]
    end
    
    subgraph Servers["MCP Server Ecosystem"]
        GH[GitHub Servers]
        SM[Smithery Servers]
        SO[mcp.so Servers]
        MS[mcpservices.com]
        MH[MCP Hubs]
        CS[Custom Servers]
    end
    
    subgraph Providers["Capability Providers"]
        TOOLS[Tools:<br/>APIs, Functions]
        RES[Resources:<br/>Files, Data]
    end
    
    Clients -->|JSON-RPC 2.0<br/>stdio/WebSocket/HTTP| Protocol
    Protocol --> Servers
    Servers --> Providers
    
    style Clients fill:#e1f5ff,color:#000000
    style Protocol fill:#fff4e1,color:#000000
    style Servers fill:#e8f5e9,color:#000000
    style Providers fill:#fce4ec,color:#000000
```

---

## 5.3 Hands-on Learning

### 5.3.1 Simplified Architecture (Educational Demo)

> **Note**: This notebook provides an educational approximation using Flask/HTTP.
> Real MCP uses JSON-RPC 2.0 over stdio/WebSocket with defined methods like 
> `initialize`, `tools/list`, `tools/call`, `resources/list`, and capability negotiation.

```mermaid
graph LR
    Client[Agent<br/>Client] -->|JSON over HTTP<br/>tools/list, execute| Server[MCP-like<br/>Tool Server]
    Server -->|results| Client
    
    Server --> Tools[tools:<br/>add, search, read_file]
    
    style Client fill:#e1f5ff,color:#000000
    style Server fill:#fff4e1,color:#000000
    style Tools fill:#e8f5e9,color:#000000
```

### 5.3.2 Minimal MCP-like Tool Server (Flask)

Run the cell below to start a simple tool server inside the notebook.
It exposes: `/capabilities`, `/tools`, `/execute`.

- add: integer addition
- search: mock search returning canned results
- read_file: reads a file under a safe demo folder

In [5]:
import threading, time, os, json
from flask import Flask, request, jsonify
from pathlib import Path

app = Flask(__name__)
SAFE_DIR = Path('mcp_demo_data')
SAFE_DIR.mkdir(exist_ok=True)
(SAFE_DIR / 'hello.txt').write_text('Hello from MCP-like server!')

TOOLS = {
    'add': {
        'description': 'Add two integers',
        'parameters': {'a': 'int', 'b': 'int'}
    },
    'search': {
        'description': 'Mock search returning example results',
        'parameters': {'query': 'string'}
    },
    'read_file': {
        'description': 'Read a text file within mcp_demo_data',
        'parameters': {'path': 'string'}
    }
}

@app.get('/capabilities')
def capabilities():
    return jsonify({'tools': True, 'resources': True})

@app.get('/tools')
def list_tools():
    return jsonify({'tools': TOOLS})

@app.post('/execute')
def execute():
    payload = request.get_json(force=True)
    name = payload.get('name')
    args = payload.get('args', {})
    try:
        if name == 'add':
            a, b = int(args.get('a', 0)), int(args.get('b', 0))
            return jsonify({'ok': True, 'result': a + b})
        elif name == 'search':
            q = str(args.get('query', '')).strip()
            return jsonify({'ok': True, 'result': [f'Result for {q} #1', f'Result for {q} #2']})
        elif name == 'read_file':
            p = Path(args.get('path', ''))
            full = (SAFE_DIR / p.name).resolve()
            if not str(full).startswith(str(SAFE_DIR.resolve())):
                return jsonify({'ok': False, 'error': 'Path not allowed'})
            if not full.exists():
                return jsonify({'ok': False, 'error': 'File not found'})
            return jsonify({'ok': True, 'result': full.read_text()})
        else:
            return jsonify({'ok': False, 'error': 'Unknown tool'})
    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)})

def run_server():
    app.run(host='127.0.0.1', port=5055, debug=False, use_reloader=False)

# Start background thread if not already running
if 'MCP_SERVER_THREAD' not in globals():
    MCP_SERVER_THREAD = threading.Thread(target=run_server, daemon=True)
    MCP_SERVER_THREAD.start()
    time.sleep(1)
    print('✅ MCP-like server running on http://127.0.0.1:5055')
else:
    print('Server already running')

Server already running


### 5.3.3 MCP-like Client: Discover & Execute

The client discovers capabilities and available tools, then executes them with arguments.

In [6]:
import requests

BASE = 'http://127.0.0.1:5055'

caps = requests.get(f'{BASE}/capabilities').json()
tools = requests.get(f'{BASE}/tools').json()['tools']
print('Capabilities:', caps)
print('Tools:', list(tools.keys()))

res_add = requests.post(f'{BASE}/execute', json={'name': 'add', 'args': {'a': 2, 'b': 40}}).json()
print('add result:', res_add)

res_read = requests.post(f'{BASE}/execute', json={'name': 'read_file', 'args': {'path': 'hello.txt'}}).json()
print('read_file result:', res_read)

Capabilities: {'resources': True, 'tools': True}
Tools: ['add', 'read_file', 'search']
add result: {'ok': True, 'result': 42}
read_file result: {'ok': True, 'result': 'Hello from MCP-like server!'}


### 5.3.4 Integrating a Tool Call into an Agent Flow

We simulate how an orchestrator could detect a tool directive and route it to the MCP server.
Format: `TOOL(name:args_json)` e.g., `TOOL(add:{"a":3,"b":5})`.

In [7]:
import json, re, requests

TOOL_RE = re.compile(r'TOOL\(([^:]+):(.+)\)')

def maybe_call_tool(text: str):
    m = TOOL_RE.search(text)
    if not m:
        return None
    name = m.group(1).strip()
    try:
        args = json.loads(m.group(2))
    except json.JSONDecodeError:
        return '[tool error: bad args json]'
    r = requests.post(f'{BASE}/execute', json={'name': name, 'args': args}).json()
    return r

print(maybe_call_tool('Try TOOL(add:{"a":7,"b":8})'))

{'ok': True, 'result': 15}


## 5.6 Production MCP: Real Specification and Implementation

Now that you understand the concepts through hands-on practice, let's learn about the **real MCP specification** used in production systems.

---

## 5.7 Practical MCP Setup and Usage

Now let's learn how to **actually use MCP** in real-world scenarios: configuring servers in IDEs and using MCP programmatically in your code.

---

## 5.8 MCP Protocol Stack: Complete Architecture

### 5.8.1 MCP Stack Layers (Systematic View)

```mermaid
graph TB
    subgraph L7["Layer 7: APPLICATION LAYER"]
        CA[Claude.ai]
        VC[VS Code]
        CU[Cursor]
        CC[Claude Code]
        JB[JetBrains IDEs]
        AG[Agents]
    end
    
    subgraph L6["Layer 6: CLIENT SDK LAYER"]
        SDK1[MCP Client Libraries<br/>Python, TypeScript, etc.]
    end
    
    subgraph L5["Layer 5: PROTOCOL SEMANTICS"]
        INIT[initialize]
        TL[tools/list]
        TC[tools/call]
        RL[resources/list]
        PL[prompts/list]
    end
    
    subgraph L4["Layer 4: JSON-RPC 2.0 ENVELOPE"]
        JSON[jsonrpc, id, method<br/>params, result, error]
    end
    
    subgraph L3["Layer 3: TRANSPORT LAYER"]
        STDIO[stdio]
        WS[WebSocket]
        HTTP[HTTP]
    end
    
    subgraph L2["Layer 2: SERVER SDK LAYER"]
        SDK2[MCP Server Libraries<br/>Python, TypeScript, etc.]
    end
    
    subgraph L1["Layer 1: CAPABILITY PROVIDERS"]
        TOOLS[Tools:<br/>APIs, Functions]
        RES[Resources:<br/>Files, Databases]
        PROMPTS[Prompts:<br/>Templates & Samples]
    end
    
    L7 --> L6
    L6 --> L5
    L5 --> L4
    L4 --> L3
    L3 --> L2
    L2 --> L1
    
    style L7 fill:#e3f2fd,color:#000000
    style L6 fill:#e1f5ff,color:#000000
    style L5 fill:#fff4e1,color:#000000
    style L4 fill:#f3e5f5,color:#000000
    style L3 fill:#e8f5e9,color:#000000
    style L2 fill:#e1f5ff,color:#000000
    style L1 fill:#fce4ec,color:#000000
```

### 5.8.2 MCP in AI System Stack Context

```mermaid
graph TB
    L6["Layer 6: Communication<br/>Message formatting, user interfaces<br/>Output rendering, streaming"]
    L5["Layer 5: Capability Integration<br/><b>MCP OPERATES HERE</b><br/>MCP Protocol<br/>Tool discovery & invocation<br/>Resource access<br/>External system integration"]
    L4["Layer 4: Coordination<br/>Multi-agent orchestration<br/>Task decomposition<br/>Workflow management"]
    L3["Layer 3: Memory<br/>Conversation history<br/>Context windows<br/>State management"]
    L2["Layer 2: Invocation<br/>API abstraction<br/>Multi-provider routing<br/>Request shaping"]
    L1["Layer 1: Execution<br/>Model providers<br/>Model inference"]
    
    L6 --> L5
    L5 --> L4
    L4 --> L3
    L3 --> L2
    L2 --> L1
    
    style L5 fill:#ffeb3b,stroke:#f57f17,stroke-width:3px,color:#000000
    style L6 fill:#e3f2fd,color:#000000
    style L4 fill:#e8f5e9,color:#000000
    style L3 fill:#fff3e0,color:#000000
    style L2 fill:#f3e5f5,color:#000000
    style L1 fill:#e0f2f1,color:#000000
```

### 5.8.3 Key Differences: Educational Demo vs Real MCP

| Aspect | Educational Demo (This Notebook) | Real MCP (Production) |
|--------|----------------------------------|------------------------|
| **Transport** | HTTP (Flask) | stdio, WebSocket, HTTP |
| **Protocol** | Custom REST endpoints | JSON-RPC 2.0 |
| **Methods** | `/capabilities`, `/tools`, `/execute` | `initialize`, `tools/list`, `tools/call` |
| **Message Format** | Simple JSON | JSON-RPC envelope (`id`, `method`, `params`) |
| **Capability Negotiation** | Basic boolean flags | Full negotiation with versioning |
| **Error Handling** | Custom `{ok, error}` | Standard JSON-RPC error objects |
| **Resource Discovery** | Not implemented | `resources/list` with URI schemes |
| **Prompt Templates** | Not implemented | `prompts/list` and `prompts/get` |

### 5.8.4 Migration Path: From Demo to Production

```mermaid
graph TD
    A[Educational Demo<br/>Flask/HTTP] -->|1. Replace HTTP with<br/>stdio/WebSocket| B[Transport Layer]
    B -->|2. Wrap messages in<br/>JSON-RPC 2.0 envelopes| C[JSON-RPC Layer]
    C -->|3. Implement<br/>initialize handshake| D[Capability Negotiation]
    D -->|4. Add resources/list<br/>and prompts/list| E[Full MCP Compliance]
    
    style A fill:#fff4e1,color:#000000
    style B fill:#e1f5ff,color:#000000
    style C fill:#f3e5f5,color:#000000
    style D fill:#e8f5e9,color:#000000
    style E fill:#c8e6c9,stroke:#4caf50,stroke-width:3px,color:#000000
```

---

## 5.9 Practice Exercises

Now that you've learned the theory, built a demo, and seen production MCP, try these exercises to reinforce your understanding:

1. Add a new tool `multiply(a,b)` and expose it in `/tools` and `/execute`.
2. Add a `/resources` endpoint serving URIs like `file://mcp_demo_data/hello.txt`.
3. Extend the `maybe_call_tool` to support `SEARCH` queries by calling the `search` tool.
4. Add a permission check: only allow `read_file` for whitelisted filenames.
5. Implement JSON-RPC style envelopes for `/execute` with `id` and `error`.

## 5.7 Practical MCP Setup and Usage

### 5.7.1 IDE Configuration: Setting Up MCP Servers

MCP adoption typically involves **configuring MCP servers in your IDE** through JSON configuration files or IDE-specific settings. The configuration method varies by IDE:

#### **Cursor IDE** (JSON Configuration File)

Cursor uses a JSON configuration file located at `~/.cursor/mcp.json` (or `%APPDATA%\Cursor\mcp.json` on Windows):

```json
{
  "mcpServers": {
    "server-name": {
      "command": "node",
      "args": ["path/to/server.js"],
      "env": {
        "API_KEY": "your-api-key"
      }
    },
    "huggingface": {
      "url": "https://huggingface.co/mcp",
      "headers": {
        "Authorization": "Bearer YOUR_TOKEN"
      }
    },
    "filesystem": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-filesystem", "/allowed/path"]
    }
  }
}
```

**Configuration Fields:**
- `command`: Executable to run (e.g., `node`, `python`, `npx`)
- `args`: Arguments passed to the command
- `env`: Environment variables for the server process
- `url`: For HTTP/WebSocket-based servers
- `headers`: HTTP headers for authentication

#### **VS Code** (Extension Configuration)

VS Code uses the **MCP Extension** with settings in `.vscode/settings.json` or user settings:

```json
{
  "mcp.servers": {
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": {
        "GITHUB_PERSONAL_ACCESS_TOKEN": "your-token"
      }
    }
  }
}
```

#### **Claude Code / JetBrains IDEs** (Plugin Settings)

These IDEs typically use plugin-based configuration through their settings UI:
1. Open IDE Settings/Preferences
2. Navigate to MCP/Claude Code settings
3. Add server configurations through the UI or import JSON

#### **Claude.ai** (Web Interface)

Claude.ai uses a web-based configuration interface where you:
1. Navigate to Settings → MCP Servers
2. Add server configurations through the web UI
3. Configure authentication tokens and server endpoints

### 5.7.2 Programmatic MCP Usage in Code

To use MCP programmatically in your code, you need an **MCP client library**. Here are examples for different languages:

#### **Python: Using MCP SDK**

```python
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import asyncio

async def use_mcp_server():
    # Configure server parameters
    server_params = StdioServerParameters(
        command="node",
        args=["path/to/mcp-server.js"],
        env={"API_KEY": "your-key"}
    )
    
    # Create client session
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Initialize connection
            await session.initialize()
            
            # List available tools
            tools = await session.list_tools()
            print(f"Available tools: {[t.name for t in tools.tools]}")
            
            # Call a tool
            result = await session.call_tool(
                name="add",
                arguments={"a": 5, "b": 3}
            )
            print(f"Result: {result.content}")
            
            # List resources
            resources = await session.list_resources()
            print(f"Available resources: {[r.uri for r in resources.resources]}")

# Run the async function
asyncio.run(use_mcp_server())
```

#### **TypeScript/JavaScript: Using MCP SDK**

```typescript
import { Client } from "@modelcontextprotocol/sdk/client/index.js";
import { StdioClientTransport } from "@modelcontextprotocol/sdk/client/stdio.js";

async function useMCP() {
  // Create transport
  const transport = new StdioClientTransport({
    command: "node",
    args: ["path/to/mcp-server.js"],
    env: { API_KEY: "your-key" }
  });

  // Create client
  const client = new Client({
    name: "my-client",
    version: "1.0.0"
  }, {
    capabilities: {}
  });

  // Connect
  await client.connect(transport);

  // List tools
  const tools = await client.listTools();
  console.log("Tools:", tools.tools.map(t => t.name));

  // Call tool
  const result = await client.callTool({
    name: "add",
    arguments: { a: 5, b: 3 }
  });
  console.log("Result:", result.content);

  // Cleanup
  await client.close();
}

useMCP();
```

#### **Direct JSON-RPC Communication (Low-Level)**

For custom implementations without SDKs:

```python
import json
import subprocess
import sys

class MCPClient:
    def __init__(self, command, args, env=None):
        self.process = subprocess.Popen(
            [command] + args,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            env=env,
            text=True
        )
        self.request_id = 0
    
    def send_request(self, method, params):
        self.request_id += 1
        request = {
            "jsonrpc": "2.0",
            "id": self.request_id,
            "method": method,
            "params": params
        }
        self.process.stdin.write(json.dumps(request) + "\n")
        self.process.stdin.flush()
        
        # Read response
        response_line = self.process.stdout.readline()
        return json.loads(response_line)
    
    def initialize(self):
        return self.send_request("initialize", {
            "clientInfo": {"name": "my-client", "version": "1.0.0"},
            "capabilities": {"tools": True, "resources": True}
        })
    
    def list_tools(self):
        return self.send_request("tools/list", {})
    
    def call_tool(self, name, arguments):
        return self.send_request("tools/call", {
            "name": name,
            "arguments": arguments
        })

# Usage
client = MCPClient("node", ["server.js"], env={"API_KEY": "key"})
client.initialize()
tools = client.list_tools()
result = client.call_tool("add", {"a": 5, "b": 3})
print(result)
```

### 5.7.3 Common MCP Usage Patterns

#### **Pattern 1: Tool Discovery and Execution**

```python
async def discover_and_use_tools(session):
    # 1. Discover available tools
    tools_response = await session.list_tools()
    available_tools = {t.name: t for t in tools_response.tools}
    
    # 2. Check if desired tool exists
    if "read_file" in available_tools:
        tool = available_tools["read_file"]
        print(f"Tool found: {tool.description}")
        
        # 3. Execute tool with proper arguments
        result = await session.call_tool(
            name="read_file",
            arguments={"path": "example.txt"}
        )
        return result.content
    else:
        raise ValueError("Tool not available")
```

#### **Pattern 2: Resource Access**

```python
async def access_resources(session):
    # 1. List available resources
    resources = await session.list_resources()
    
    # 2. Access specific resource by URI
    for resource in resources.resources:
        if resource.uri.startswith("file://"):
            # Read resource content
            content = await session.read_resource(resource.uri)
            print(f"Resource {resource.name}: {content}")
```

#### **Pattern 3: Error Handling and Retries**

```python
async def robust_tool_call(session, tool_name, arguments, max_retries=3):
    for attempt in range(max_retries):
        try:
            result = await session.call_tool(
                name=tool_name,
                arguments=arguments
            )
            return result
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            print(f"Attempt {attempt + 1} failed: {e}, retrying...")
            await asyncio.sleep(2 ** attempt)  # Exponential backoff
```

#### **Pattern 4: Multi-Server Orchestration**

```python
async def orchestrate_multiple_servers(servers_config):
    sessions = []
    
    # Connect to multiple servers
    for config in servers_config:
        transport = create_transport(config)
        session = ClientSession(transport)
        await session.initialize()
        sessions.append(session)
    
    # Use tools from different servers
    results = []
    for session in sessions:
        tools = await session.list_tools()
        if "process_data" in [t.name for t in tools.tools]:
            result = await session.call_tool("process_data", {...})
            results.append(result)
    
    # Cleanup
    for session in sessions:
        await session.close()
    
    return results
```

### 5.7.4 Installation and Setup Steps

**1. Install MCP Server:**
```bash
# Using npm (for Node.js servers)
npm install -g @modelcontextprotocol/server-filesystem

# Using pip (for Python servers)
pip install mcp-server-filesystem

# Or use npx to run without installation
npx -y @modelcontextprotocol/server-filesystem
```

**2. Configure IDE:**
- Copy server configuration JSON to IDE config file
- Set environment variables (API keys, tokens)
- Restart IDE to load configuration

**3. Verify Connection:**
- Check IDE logs for connection status
- Test tool discovery in IDE's MCP panel
- Try calling a tool to verify functionality

**4. Use in Code:**
- Import MCP client SDK
- Create client session with server configuration
- Discover and call tools programmatically

---



## 5.6 Production MCP: Real Specification and Implementation

### 5.6.1 Anthropic's Role in MCP Ecosystem

**Anthropic** is the originator and primary maintainer of the Model Context Protocol (MCP) standard. They introduced MCP to:

1. **Standardize** tool/resource discovery and invocation across AI systems
2. **Enable** plug-and-play capability integration
3. **Support** capability negotiation and secure scoping
4. **Decouple** agent implementations from tool providers

### 5.6.2 Production MCP Specification

Real MCP implementations follow the official specification with:

- **Transport**: JSON-RPC 2.0 messages over stdio, WebSocket, or HTTP
- **Protocol**: Strict JSON-RPC 2.0 envelope format
- **Methods**: Standardized method names (`initialize`, `tools/list`, `tools/call`, etc.)
- **Capabilities**: Negotiated feature sets (tools, resources, prompts)
- **Security**: Permission scoping and provenance tracking

### 5.6.3 Real-World Usage Patterns

| Use Case | Client | Server Provider | Example |
|----------|--------|-----------------|---------|
| **IDE Integration** | VS Code, Cursor | GitHub MCP servers | File operations, Git integration |
| **Agent Tool Access** | Custom agents | Smithery marketplace | API integrations, data access |
| **Enterprise** | Internal tools | Custom servers | Company-specific capabilities |
| **Development** | CLI tools | mcp.so registry | Testing, development workflows |

### 5.6.4 Transport Options

Production MCP supports multiple transports:

1. **stdio** (most common): Direct process communication, ideal for local servers
2. **WebSocket**: Network-based, supports remote servers and real-time updates
3. **HTTP**: REST-like interface, easier for web integration

### 1) initialize (handshake)
Client → Server
```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "initialize",
  "params": {
    "clientInfo": { "name": "demo-agent", "version": "0.1.0" },
    "capabilities": {
      "tools": true,
      "resources": true
    }
  }
}
```

Client ← Server
```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "serverInfo": { "name": "example-mcp-server", "version": "1.0.0" },
    "capabilities": {
      "tools": true,
      "resources": true
    }
  }
}
```

### 2) tools/list
Client → Server
```json
{
  "jsonrpc": "2.0",
  "id": 2,
  "method": "tools/list",
  "params": {}
}
```

Client ← Server
```json


{
  "jsonrpc": "2.0",
  "id": 2,
  "result": {
    "tools": [
      {
        "name": "add",
        "description": "Add two integers",
        "parameters": {
          "type": "object",
          "properties": {
            "a": {"type": "integer"},
            "b": {"type": "integer"}
          },
          "required": ["a", "b"]
        }
      },
      {
        "name": "read_file",
        "description": "Read a text file",
        "parameters": {
          "type": "object",
          "properties": {
            "path": {"type": "string"}
          },
          "required": ["path"]
        }
      }
    ]
  }
}
```

### 3) tools/call
Client → Server
```json
{
  "jsonrpc": "2.0",
  "id": 3,
  "method": "tools/call",
  "params": {
    "name": "add",
    "arguments": { "a": 3, "b": 5 }
  }
}
```

Client ← Server
```json
{
  "jsonrpc": "2.0",
  "id": 3,
  "result": {
    "ok": true,
    "result": 8
  }
}
```

### 4) resources/list (optional)
Client → Server
```json
{
  "jsonrpc": "2.0",
  "id": 4,
  "method": "resources/list",
  "params": {}
}
```

Client ← Server
```json
{
  "jsonrpc": "2.0",
  "id": 4,
  "result": {
    "resources": [
      {
        "uri": "file://mcp_demo_data/hello.txt",
        "name": "hello.txt",
        "mime": "text/plain"
      }
    ]
  }
}
```
---

### 5.6.5 Optional: Minimal Python WebSocket Client (JSON-RPC)


This snippet shows how a client could speak MCP‑style JSON‑RPC to a server over WebSocket.


It won’t run unless you have a compatible MCP server listening (e.g., ws://127.0.0.1:5056).


```python
import asyncio, json, websockets

async def mcp_client():
    async with websockets.connect("ws://127.0.0.1:5056") as ws:
        # 1) initialize
        init_msg = {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "initialize",
            "params": {
                "clientInfo": {"name": "demo-agent", "version": "0.1.0"},
                "capabilities": {"tools": True, "resources": True}
            }
        }
        await ws.send(json.dumps(init_msg))
        print("--> sent initialize")
        print("<--", await ws.recv())

        # 2) list tools
        list_msg = {"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}}
        await ws.send(json.dumps(list_msg))
        print("--> sent tools/list")
        tools_resp = json.loads(await ws.recv())
        print("<--", tools_resp)
        first_tool = tools_resp.get("result", {}).get("tools", [{}])[0].get("name")

        # 3) call first tool if exists
        if first_tool == "add":
            call_msg = {
                "jsonrpc": "2.0",
                "id": 3,
                "method": "tools/call",
                "params": {
                    "name": "add",
                    "arguments": {"a": 10, "b": 32}
                }
            }
            await ws.send(json.dumps(call_msg))
            print("--> sent tools/call add")
            print("<--", await ws.recv())

asyncio.run(mcp_client())


```


### Key Points in Real MCP
1. Strict JSON‑RPC 2.0 envelope (`jsonrpc`, `id`, `method`, `params`, `result`, `error`).
2. Capability negotiation lets servers and clients agree on supported features.
3. Tool schemas use JSON Schema for argument validation.
4. Resource URIs can represent files, databases, or external APIs.
5. Provenance and permission layers can restrict tool usage.
6. Extensible to streaming responses and cancellation tokens.


Use these structures to evolve the earlier Flask demo toward true MCP compliance when you adopt a production library.

## 5.10 MCP Summary: Systematic Integration

### 5.10.1 MCP in the Complete System Context

MCP serves as the **standardized bridge** between AI agents (Layer 4-6) and external capabilities:

```mermaid
graph TB
    L6["Layer 6: Communication<br/>Formats MCP tool results<br/>for user presentation"]
    L5["Layer 5: Capability Integration<br/><b>MCP PROTOCOL</b><br/>Client: Agents/IDEs discover tools via MCP<br/>Protocol: JSON-RPC 2.0<br/>Server: Tool providers expose capabilities"]
    L4["Layer 4: Coordination<br/>Orchestrates MCP tool calls<br/>in multi-agent workflows"]
    L3["Layer 3: Memory<br/>Stores tool call history<br/>and results"]
    L2["Layer 2: Invocation<br/>May use MCP tools<br/>to enhance model context"]
    L1["Layer 1: Execution<br/>Models execute<br/>may trigger MCP tool calls"]
    
    L6 --> L5
    L5 --> L4
    L4 --> L3
    L3 --> L2
    L2 --> L1
    
    style L5 fill:#ffeb3b,stroke:#f57f17,stroke-width:3px,color:#000000
    style L6 fill:#e3f2fd,color:#000000
    style L4 fill:#e8f5e9,color:#000000
    style L3 fill:#fff3e0,color:#000000
    style L2 fill:#f3e5f5,color:#000000
    style L1 fill:#e0f2f1,color:#000000
```

### 5.10.2 Key Takeaways

1. **Hierarchical Position**: MCP operates at Layer 5 (Capability Integration), bridging agents with external tools/resources

2. **Ecosystem Structure**:
   - **Origin**: Anthropic (standard creator and maintainer)
   - **Clients**: Claude.ai (web), Claude Code, VS Code, Cursor, JetBrains IDEs, custom agent runtimes
   - **Servers**: GitHub, Smithery, mcp.so, mcpservices.com, MCP Hubs, custom servers

3. **Protocol Stack**:
   - Application Layer (IDEs/Agents)
   - Client SDK Layer
   - Protocol Semantics (initialize, tools/list, tools/call, resources/list)
   - JSON-RPC 2.0 Envelope
   - Transport Layer (stdio/WebSocket/HTTP)
   - Server SDK Layer
   - Capability Providers (Tools/Resources/Prompts)

4. **Theoretical Foundation**:
   - Standardized discovery and invocation
   - Capability negotiation
   - Decoupled architecture
   - Secure scoping and permissions

5. **Production vs Educational**:
   - Production uses JSON-RPC 2.0 with stdio/WebSocket
   - Educational demo uses HTTP/REST for simplicity
   - Migration path clearly defined

### 5.10.3 MCP Ecosystem Flow

```mermaid
graph TD
    ANTH[Anthropic<br/>Standard Creator]
    
    ANTH -->|Web Client| CA[Claude.ai]
    ANTH -->|IDE Clients| IDES[VS Code<br/>Cursor<br/>Claude Code<br/>JetBrains IDEs]
    ANTH -->|Custom Clients| AGENTS[Agent Runtimes]
    
    CA -->|MCP Protocol| SERVERS[MCP Server Ecosystem]
    IDES -->|MCP Protocol| SERVERS
    AGENTS -->|MCP Protocol| SERVERS
    
    SERVERS --> GH[GitHub Servers]
    SERVERS --> SM[Smithery Servers]
    SERVERS --> SO[mcp.so Servers]
    SERVERS --> MS[mcpservices.com]
    SERVERS --> MH[MCP Hubs]
    SERVERS --> CS[Custom Servers]
    
    style ANTH fill:#ffeb3b,stroke:#f57f17,stroke-width:3px,color:#000000
    style CA fill:#e1f5ff,color:#000000
    style IDES fill:#e1f5ff,color:#000000
    style AGENTS fill:#e1f5ff,color:#000000
    style SERVERS fill:#e8f5e9,color:#000000
    style GH fill:#fce4ec,color:#000000
    style SM fill:#fce4ec,color:#000000
    style SO fill:#fce4ec,color:#000000
    style MS fill:#fce4ec,color:#000000
    style MH fill:#fce4ec,color:#000000
    style CS fill:#fce4ec,color:#000000
```

---

## 5.11 MCP vs A2A: Understanding the Differences

| Dimension | MCP (Model Context Protocol) | A2A (Agent-to-Agent Messaging) |
|-----------|------------------------------|--------------------------------|
| Primary Purpose | Standardize tool/resource discovery & invocation | Exchange arbitrary messages (plans, critiques, data) between autonomous agents |
| Abstraction Level | Tool/service integration layer | Conversation / coordination / workflow layer |
| Communication Style | Request/Response (JSON-RPC) | Event/message driven (queue, pub/sub, broker) |
| Typical Transport | stdio, WebSocket, pipes | Pub/Sub, WebSocket, HTTP, in-memory queues |
| Message Envelope | `jsonrpc`, `id`, `method`, `params`, `result`/`error` | Custom envelope: `id`, `sender`, `recipient`, `type`, `content`, `context`, `provenance` |
| Discovery | Built-in via `tools/list`, `resources/list` | Ad-hoc (registry service), topic subscriptions, static config |
| Capability Negotiation | Yes (`initialize` returns capabilities) | Usually no formal negotiation; can introduce handshake protocol |
| Validation | JSON Schema for tool params | Optional (custom schema / pydantic) |
| Streaming | Supported by MCP extensions (tool call streaming) | Depends on channel; WebSocket / streaming Pub/Sub |
| State Management | External; MCP itself is stateless | Agents may maintain internal memory; broker can persist |
| Security Focus | Tool permission, resource access scope | Identity, authorization, provenance, rate limiting |
| Error Handling | Standard JSON-RPC error object | Custom (ACK/NACK, retries, dead-letter) |
| Best For | Integrating computed functions, file access, retrieval APIs | Coordinating multi-role reasoning across distributed agent nodes |
| Scaling Pattern | Add more servers exposing tools; clients discover | Add more agents subscribing/publishing messages; horizontal scaling |
| Latency Profile | Synchronous call latency (tool execution time) | Potential additional transit latency; can batch/parallelize |
| Composability | Agents + MCP servers + tool ecosystems | Orchestrators + message bus + optional MCP for tool calls |
| Example Use | "Summarize file X" via `read_file` + `summarize` tool | Planner sends plan to researcher; critic evaluates synthesis |
| Interop | Shared protocol allows plug-and-play servers | Custom; varies by message schema agreements |
| Typical Failure Modes | Tool not found, invalid params, resource unavailable | Message loss, duplication, ordering issues, backpressure |
| Complementarity | Provides structured tool invocation inside A2A workflows | Wraps MCP tool results inside broader multi-agent dialogue |

### When to Use Which?
- Use **MCP** when you need a consistent method to *discover and invoke tools/resources* regardless of implementation details.
- Use **A2A** when you need *agents to reason, negotiate, and coordinate* over evolving state/contexts.
- Combine both: Agents exchange envelopes (A2A) and invoke tools through MCP when they detect a need.

### Integration Pattern

```mermaid
sequenceDiagram
    participant A2A as Agent Message Bus<br/>(A2A)
    participant MCPC as MCP Client
    participant MCPS as MCP Server
    
    A2A->>MCPC: envelope content contains TOOL(...)
    MCPC->>MCPS: tools/call(name, arguments)
    MCPS-->>MCPC: result({ok, result})
    MCPC-->>A2A: results appended as new A2A envelope
```
